# NMF on ModulAir 00691

In [2]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [3]:
#importing data from Modulair MOD-00691
data = pd.read_csv('MOD-00691.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:34Z,577611104,2025-12-31T18:59:34Z,MOD-00691,51.5,0.0,6.169,0.652,0.217,0.057,0.027,...,37.685,24.425,14346.0,14347.0,14348.0,14478.0,14503.0,14553.0,14528.0,1.35
2025-12-31T23:58:34Z,577611106,2025-12-31T18:58:34Z,MOD-00691,51.4,0.0,6.513,0.912,0.194,0.036,0.022,...,36.979,24.792,14346.0,14347.0,14348.0,14478.0,14503.0,14553.0,14528.0,0.86
2025-12-31T23:57:34Z,577611105,2025-12-31T18:57:34Z,MOD-00691,51.2,0.0,6.793,0.800,0.232,0.062,0.022,...,36.276,25.170,14346.0,14347.0,14348.0,14478.0,14503.0,14553.0,14528.0,2.51
2025-12-31T23:56:34Z,577611107,2025-12-31T18:56:34Z,MOD-00691,51.8,0.0,6.516,0.755,0.209,0.047,0.047,...,37.199,24.746,14346.0,14347.0,14348.0,14478.0,14503.0,14553.0,14528.0,1.39
2025-12-31T23:55:34Z,577609047,2025-12-31T18:55:34Z,MOD-00691,51.6,-0.1,6.421,0.860,0.280,0.026,0.036,...,36.963,25.476,14346.0,14347.0,14348.0,14478.0,14503.0,14553.0,14528.0,1.39


In [4]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:34Z,2025-12-31T18:59:34Z,1158.375,2.424,37.685,24.425,6.169,0.652,0.217,0.057,0.027,0.004
2025-12-31T23:58:34Z,2025-12-31T18:58:34Z,1285.293,2.293,36.979,24.792,6.513,0.912,0.194,0.036,0.022,0.018
2025-12-31T23:57:34Z,2025-12-31T18:57:34Z,1714.006,2.812,36.276,25.170,6.793,0.800,0.232,0.062,0.022,0.018
2025-12-31T23:56:34Z,2025-12-31T18:56:34Z,2682.790,2.424,37.199,24.746,6.516,0.755,0.209,0.047,0.047,0.017
2025-12-31T23:55:34Z,2025-12-31T18:55:34Z,1489.066,2.424,36.963,25.476,6.421,0.860,0.280,0.026,0.036,0.026


In [5]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:34Z,1158.375,2.424,37.685,24.425,6.169,0.652,0.217,0.057,0.027,0.004
1,2025-12-31T18:58:34Z,1285.293,2.293,36.979,24.792,6.513,0.912,0.194,0.036,0.022,0.018
2,2025-12-31T18:57:34Z,1714.006,2.812,36.276,25.170,6.793,0.800,0.232,0.062,0.022,0.018
3,2025-12-31T18:56:34Z,2682.790,2.424,37.199,24.746,6.516,0.755,0.209,0.047,0.047,0.017
4,2025-12-31T18:55:34Z,1489.066,2.424,36.963,25.476,6.421,0.860,0.280,0.026,0.036,0.026


In [6]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:34,1158.375,2.424,37.685,24.425,6.169,0.652,0.217,0.057,0.027,0.004
1,2025-12-31 18:58:34,1285.293,2.293,36.979,24.792,6.513,0.912,0.194,0.036,0.022,0.018
2,2025-12-31 18:57:34,1714.006,2.812,36.276,25.170,6.793,0.800,0.232,0.062,0.022,0.018
3,2025-12-31 18:56:34,2682.790,2.424,37.199,24.746,6.516,0.755,0.209,0.047,0.047,0.017
4,2025-12-31 18:55:34,1489.066,2.424,36.963,25.476,6.421,0.860,0.280,0.026,0.036,0.026


In [7]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,960.72,36.11,28.14,2.03,20.59,2.42,1.15,0.45,0.63,0.53
1,2025-03-31 21:00:00,1042.71,39.51,26.46,2.66,28.44,4.94,1.70,0.54,0.74,0.63
2,2025-03-31 22:00:00,1250.15,35.64,31.83,2.79,24.14,5.45,1.66,0.46,0.58,0.46
3,2025-03-31 23:00:00,1134.17,21.40,42.59,2.79,16.63,5.03,1.41,0.32,0.32,0.20
4,2025-04-01 00:00:00,1275.50,25.50,45.24,2.73,16.34,3.76,1.07,0.24,0.25,0.15
...,...,...,...,...,...,...,...,...,...,...,...
6545,2025-12-31 14:00:00,1195.91,35.29,32.69,2.04,6.81,0.61,0.18,0.03,0.03,0.02
6546,2025-12-31 15:00:00,1023.29,35.38,32.82,1.99,6.27,0.61,0.17,0.03,0.03,0.02
6547,2025-12-31 16:00:00,1021.94,36.26,30.29,2.22,7.37,0.80,0.22,0.04,0.03,0.02
6548,2025-12-31 17:00:00,946.88,37.36,26.09,2.02,7.94,0.93,0.26,0.05,0.04,0.02


In [8]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,960.72,36.11,28.14,2.03,20.59,2.42,1.15,0.45,0.63,0.53
2025-03-31 21:00:00,1042.71,39.51,26.46,2.66,28.44,4.94,1.70,0.54,0.74,0.63
2025-03-31 22:00:00,1250.15,35.64,31.83,2.79,24.14,5.45,1.66,0.46,0.58,0.46
2025-03-31 23:00:00,1134.17,21.40,42.59,2.79,16.63,5.03,1.41,0.32,0.32,0.20
2025-04-01 00:00:00,1275.50,25.50,45.24,2.73,16.34,3.76,1.07,0.24,0.25,0.15


In [9]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [10]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [11]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.336376,0.704723,0.314695,0.053379,0.324559,0.086090,0.090480,0.119048,0.190909,0.33125
2025-03-31 21:00:00,0.365083,0.771077,0.295907,0.069945,0.448298,0.175738,0.133753,0.142857,0.224242,0.39375
2025-03-31 22:00:00,0.437714,0.695550,0.355961,0.073363,0.380517,0.193881,0.130606,0.121693,0.175758,0.28750
2025-03-31 23:00:00,0.397106,0.417642,0.476292,0.073363,0.262137,0.178940,0.110936,0.084656,0.096970,0.12500
2025-04-01 00:00:00,0.446590,0.497658,0.505927,0.071785,0.257566,0.133760,0.084186,0.063492,0.075758,0.09375


In [12]:
data.to_csv(r'MOD-00691_timeseries_hourly_scaled.csv')

In [15]:
start = data.index.min()

end = data.index.min()

print(start, end)

2025-03-31 20:00:00 2025-03-31 20:00:00


## Implementing NMF

In [16]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [17]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.342524,0.704013,0.321130,0.066097,0.273786,0.199207,0.168617,0.148345,0.164090,0.199962
2025-03-31 21:00:00,0.386407,0.767172,0.298984,0.071958,0.397531,0.271455,0.220565,0.189624,0.205387,0.247236
2025-03-31 22:00:00,0.386827,0.709143,0.375086,0.073984,0.373741,0.232759,0.182595,0.154607,0.168182,0.204560
2025-03-31 23:00:00,0.328803,0.434964,0.495060,0.067777,0.289645,0.152234,0.112325,0.092621,0.103521,0.129367
2025-04-01 00:00:00,0.358508,0.519661,0.530408,0.073444,0.284073,0.127116,0.084584,0.066076,0.074527,0.096565
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.324436,0.711766,0.391144,0.064578,0.129278,0.034173,0.012838,0.006645,0.011040,0.022970
2025-12-31 15:00:00,0.312833,0.701344,0.378743,0.062495,0.109521,0.027905,0.010483,0.005784,0.010609,0.022408
2025-12-31 16:00:00,0.314012,0.718416,0.350534,0.061657,0.126982,0.034435,0.012936,0.006580,0.010251,0.021486


In [18]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.052674,0.056134,0.104755,0.030420
1,0.055748,0.084044,0.132464,0.023640
2,0.052232,0.077420,0.106289,0.040795
3,0.031850,0.056461,0.061587,0.076343
4,0.039923,0.054699,0.041140,0.078973
...,...,...,...,...
6494,0.059069,0.022828,0.000000,0.042308
6495,0.058351,0.018641,0.000000,0.040368
6496,0.059623,0.023003,0.000000,0.034136
6497,0.060972,0.024962,0.000357,0.023294


In [19]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [20]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,3.835217,11.875290,2.893661,0.728806,0.167133,0.000000,0.000000,0.027275,0.097821,0.246889
Feature 2,1.387926,0.451268,0.071347,0.182059,4.509721,1.496986,0.562378,0.194625,0.037892,0.000000
Feature 3,0.143109,0.507451,0.071944,0.047713,0.000000,1.099477,1.308274,1.294051,1.466750,1.727154
Feature 4,1.564941,0.000000,5.166615,0.410600,0.389036,0.000000,0.000000,0.013977,0.103917,0.198218


In [21]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.052674,0.056134,0.104755,0.030420
2025-03-31 21:00:00,0.055748,0.084044,0.132464,0.023640
2025-03-31 22:00:00,0.052232,0.077420,0.106289,0.040795
2025-03-31 23:00:00,0.031850,0.056461,0.061587,0.076343
2025-04-01 00:00:00,0.039923,0.054699,0.041140,0.078973
...,...,...,...,...
2025-12-31 14:00:00,0.059069,0.022828,0.000000,0.042308
2025-12-31 15:00:00,0.058351,0.018641,0.000000,0.040368
2025-12-31 16:00:00,0.059623,0.023003,0.000000,0.034136


In [22]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,3.835217,11.875290,2.893661,0.728806,0.167133,0.000000,0.000000,0.027275,0.097821,0.246889
Factor 2,1.387926,0.451268,0.071347,0.182059,4.509721,1.496986,0.562378,0.194625,0.037892,0.000000
Factor 3,0.143109,0.507451,0.071944,0.047713,0.000000,1.099477,1.308274,1.294051,1.466750,1.727154
Factor 4,1.564941,0.000000,5.166615,0.410600,0.389036,0.000000,0.000000,0.013977,0.103917,0.198218


In [23]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.552402,0.126984,0.006827,0.294700,0.019086
no,0.508925,0.080755,0.011035,0.374868,0.024416
no2,0.965897,0.023315,0.013670,0.000000,-0.002882
o3,0.299065,0.004684,0.002463,0.698139,-0.004350
bin0,0.047610,0.816019,0.000000,0.144891,-0.008519
bin1,0.000000,0.802041,0.307140,0.000000,-0.109181
bin2,0.000000,0.477608,0.579312,0.000000,-0.056920
bin3,0.045825,0.207704,0.720059,0.030702,-0.004289
bin4,0.130712,0.032162,0.649123,0.181546,0.006457
bin5,0.225994,0.000000,0.523619,0.237222,0.013165


In [24]:
results.to_csv(r'MOD-00691_4_factor_results.csv')
comp.to_csv(r'MOD-00691_4_factor_comp.csv')
res.to_csv(r'MOD-00691_4_factor_resid.csv')

In [26]:
# check how many rows of data you have
len(data)

6499